In [ ]:
import json
import numpy as np
import os
import pandas as pd

from tqdm import tqdm

In [ ]:
# Load preprocessed patient data
df_patientdata = pd.read_csv('data/preprocessed/patient_data.csv', dtype={'stay_id': str, 'subject_id': str, 'acuity': str, 'disposition': str, 'complexity': str, 'los': float})

df_patientdata

In [ ]:
# Load preprocessed event logs
df_eventlogs = pd.read_csv('data/preprocessed/event_logs.csv', dtype={'case_id': str, 'activity_name': str})
df_eventlogs['timestamp'] = pd.to_datetime(df_eventlogs['timestamp'])

df_eventlogs

In [ ]:
# Get list of ED activities
list_activities = sorted(set(df_eventlogs['activity_name']))

list_activities

In [ ]:
# Group event logs per case ID
df_eventlogs_per_caseid = df_eventlogs.groupby(['case_id']).aggregate({'activity_name': list, 'timestamp': list}).reset_index()

df_eventlogs_per_caseid

In [ ]:
# Merge patient data with event logs
df_ehrs = pd.merge(df_patientdata, df_eventlogs_per_caseid, how='left', left_on='stay_id', right_on='case_id')
df_ehrs.dropna(inplace=True)

df_ehrs

In [ ]:
# Impose a 5-minute temporal resolution
TEMPORAL_RESOLUTION = 5  # in minutes
new_activity_col, new_timestamp_col = [], []
for idx in tqdm(df_ehrs.index):
    row = df_ehrs.loc[idx]

    activity_list, timestamp_list = row['activity_name'], row['timestamp']
    new_activity_list, new_timestamp_list = [activity_list[0]], [timestamp_list[0]]
    for act_idx, time_idx in zip(activity_list[1:], timestamp_list[1:]):
        if act_idx != new_activity_list[-1]:
            if (time_idx - new_timestamp_list[-1]).total_seconds() > 60:
                new_timestamp_list.append(time_idx)
                new_activity_list.append(act_idx)
        else:
            if (time_idx - new_timestamp_list[-1]).total_seconds() > (60 * TEMPORAL_RESOLUTION):
                new_timestamp_list.append(time_idx)
                new_activity_list.append(act_idx)

    new_timestamp_col.append(new_timestamp_list)
    new_activity_col.append(''.join(new_activity_list))

df_ehrs.loc[:, 'activity_name'] = new_activity_col
df_ehrs.loc[:, 'timestamp'] = np.array(new_timestamp_col, dtype=object)

In [ ]:
# Add column for case length
df_ehrs.loc[:, 'case_len'] = df_ehrs.activity_name.str.len()

df_ehrs

In [ ]:
# Get cases that starts with patient arrival
df_ehrs = df_ehrs[df_ehrs['activity_name'].str.startswith('A')].copy()

df_ehrs

In [ ]:
# Get cases that ends with patient discharge
df_ehrs = df_ehrs[df_ehrs['activity_name'].str.endswith('G')].copy()

df_ehrs

In [ ]:
# Get cases with at least 3 activities
df_ehrs = df_ehrs[df_ehrs['case_len'] >= 3].copy()

df_ehrs

In [ ]:
# Limit study to patients with LOS within 72 hours
df_ehrs = df_ehrs[df_ehrs['los'] <= 72].copy()

df_ehrs

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plot percentage of EHRs with the activity
x_list, y_list = [], []
for act_name in sorted(list_activities):
    df_act = df_ehrs[df_ehrs['activity_name'].str.contains(act_name)]
    print(act_name, len(df_act)/len(df_ehrs) * 100)
    x_list.append(act_name)
    y_list.append(round(len(df_act)/len(df_ehrs) * 100, 2))
ax = sns.barplot(x=x_list, y=y_list, color='blue')
ax.bar_label(ax.containers[0])
plt.show()

In [ ]:
# Identify records with temporal deviations
TEMPORAL_DEVIATION = 3  # z-score
REMOVAL_TYPE = '3std'

dict_edge_duration = {}
for edge_from in list_activities:
    for edge_to in list_activities:
        dict_edge_duration[(edge_from, edge_to)] = []

for case_time, case_act in zip(df_ehrs['timestamp'], df_ehrs['activity_name']):
    assert len(case_time) == len(case_act)

    for idx in range(1, len(case_time)):
        edge_name = (case_act[idx-1], case_act[idx])
        edge_duration = (case_time[idx] - case_time[idx-1]).total_seconds() / (60*60)

        dict_edge_duration[edge_name].append(edge_duration)

dict_edge_stats = {}
for edge_name, list_edge_durations in dict_edge_duration.items():
    if len(list_edge_durations) != 0:
        dict_edge_stats[edge_name] = {'mean': np.mean(list_edge_durations),
                                      'std': np.std(list_edge_durations),
                                      'median': np.median(list_edge_durations),
                                      'mad': np.median(np.abs(np.array(list_edge_durations) - np.median(list_edge_durations)))
                                      }

        if np.std(list_edge_durations) == 0:
            dict_edge_stats[edge_name]['std'] = 1e-10  # To prevent divide by 0
        if dict_edge_stats[edge_name]['mad'] == 0:
            dict_edge_stats[edge_name]['mad'] = 1e-10

    else:
        dict_edge_stats[edge_name] = {'mean': 0.,
                                      'std': 1e-10,
                                      'median': 0.,
                                      'mad': 0.
                                      }

list_time_diffs = []
list_zscore_normal = []
list_zscore_modified = []
list_deviation_count = []
for case_time, case_act in zip(df_ehrs['timestamp'], df_ehrs['activity_name']):
    assert len(case_time) == len(case_act)

    time_between_acts = []

    z_cnt_normal = 0
    z_cnt_mod = 0
    if TEMPORAL_DEVIATION != 0:
        case_deviation_count = 0
        for idx in range(1,len(case_time)):
            edge_name = (case_act[idx-1], case_act[idx])
            edge_duration = (case_time[idx] - case_time[idx-1]).total_seconds() / (60*60)

            modified_z_score = 0.6745 * (edge_duration - dict_edge_stats[edge_name]['median']) / dict_edge_stats[edge_name]['mad']

            if modified_z_score > (TEMPORAL_DEVIATION):
                z_cnt_mod += 1

            if modified_z_score > TEMPORAL_DEVIATION:
                if REMOVAL_TYPE == '3std':
                    time_between_acts.append(int((dict_edge_stats[edge_name]['median'] + (3*1.4826*dict_edge_stats[edge_name]['mad'])) * 60))  # in minutes
                case_deviation_count += 1
            else:
                time_between_acts.append(int(edge_duration * 60))  # in minutes

        time_between_acts = [max(1, x) for x in time_between_acts]
        list_time_diffs.append(time_between_acts)
        list_deviation_count.append(case_deviation_count)
    else:
        for idx in range(1,len(case_time)):
            edge_name = (case_act[idx-1], case_act[idx])
            edge_duration = (case_time[idx] - case_time[idx-1]).total_seconds() / (60*60)

            time_between_acts.append(int(edge_duration * 60))

        time_between_acts = [max(1, x) for x in time_between_acts]
        list_time_diffs.append(time_between_acts)
        list_deviation_count.append(0)

    list_zscore_normal.append(z_cnt_normal)
    list_zscore_modified.append(z_cnt_mod)

df_ehrs.loc[:, 'time_diffs'] = np.array(list_time_diffs, dtype=object)
df_ehrs.loc[:, 'temporal_deviation_count'] = list_deviation_count

In [ ]:
# Save frequency distributions of patient indicator
outpath_frequency = 'params/frequency'
if not os.path.exists(outpath_frequency):
    os.makedirs(outpath_frequency)

# For acuity
df_frequency = df_ehrs.groupby(['acuity']).case_id.count() / df_ehrs.case_id.count()
df_frequency = df_frequency.rename('percent')
df_frequency.to_csv(f'{outpath_frequency}/frequency.csv')

# For disposition per acuity
df_frequency = df_ehrs.groupby(['acuity', 'disposition']).case_id.count() / df_ehrs.groupby('acuity').case_id.count()
df_frequency = df_frequency.rename('percent')
df_frequency.to_csv(f'{outpath_frequency}/frequency_groupname.csv')

In [ ]:
# Save as df as pickle file for ABM simulations
import pickle

df_ehrs.to_pickle(f'syn_data/baseline_synthetic_data_{TEMPORAL_DEVIATION}_{REMOVAL_TYPE}.pkl')
df_ehrs